In [1]:
from r1_data_generating import *
import bambi as bmb
import pymc as pm
import arviz as az



 Loading processed population datasets... 




 Loading MLEs for the processed population datasets... 




In [2]:
trace_grw_f = pd.read_pickle(open(file = os.getcwd()+"/true_popu/glm_mcmc/trace_grw_f.pkl", mode="rb"))
trace_grw_nf = pd.read_pickle(open(file = os.getcwd()+"/true_popu/glm_mcmc/trace_grw_nf.pkl", mode="rb"))
trace_sur = pd.read_pickle(open(file = os.getcwd()+"/true_popu/glm_mcmc/trace_sur.pkl", mode="rb"))
trace_fec = pd.read_pickle(open(file = os.getcwd()+"/true_popu/glm_mcmc/trace_fec.pkl", mode="rb"))
trace_flow = pd.read_pickle(open(file = os.getcwd()+"/true_popu/glm_mcmc/trace_flow.pkl", mode="rb"))
theta_mle_full = pd.read_pickle(open(file = os.getcwd()+f"/true_popu/mle/theta_mle_full.pkl", mode="rb"))

Time_all = 9
for i in range(Time_all):
    globals()[f'p_all{i}'] = pickle.load(open(file = os.getcwd()+f"/true_popu/popu_dataset{i}.pkl", mode="rb"))
popu_all = {i: globals()[f'p_all{i}'] for i in range(Time_all)}
for i in range(Time_all):
    del(globals()[f'p_all{i}']) 

In [3]:
# loading glms

# grw_nf
df_grw_nf = pd.DataFrame(XY_grw_nf_compu(popu_whole, env=True)[0], columns=['size', 'age', 'tmax', 'tmin', 'precip'])
df_grw_nf['size_Next'] = XY_grw_nf_compu(popu_whole, env=True)[1]
model_grw_nf = bmb.Model("size_Next ~ size + age + tmax + tmin + precip", df_grw_nf)

# grw_f
df_grw_f = pd.DataFrame(XY_grw_f_compu(popu_whole, env=True)[0], columns=['size', 'age', 'tmax', 'tmin', 'precip'])
df_grw_f['size_Next'] = XY_grw_f_compu(popu_whole, env=True)[1]
model_grw_f = bmb.Model("size_Next ~ size + age + tmax + tmin + precip", df_grw_f)

# sur
df_sur = pd.DataFrame(XY_sur_compu(popu_whole, env=True)[0], columns=['size', 'age', 'tmax', 'tmin', 'precip'])
df_sur['size_Next'] = XY_sur_compu(popu_whole, env=True)[1]
model_sur = bmb.Model("size_Next ~ size + age + tmax + tmin + precip", df_sur, family='bernoulli')

# fec
df_fec = pd.DataFrame(XY_fec_compu(popu_whole, env=True)[0], columns=['size', 'age', 'tmax', 'tmin', 'precip'])
df_fec['size_Next'] = XY_fec_compu(popu_whole, env=True)[1]
model_fec = bmb.Model("size_Next ~ size + age + tmax + tmin + precip", df_fec, family='bernoulli')

# flow
df_flow = pd.DataFrame(XY_flow_compu(popu_whole, env=True)[0], columns=['size', 'age', 'tmax', 'tmin', 'precip'])
df_flow['size_Next'] = XY_flow_compu(popu_whole, env=True)[1]
model_flow = bmb.Model("size_Next ~ size + age + tmax + tmin + precip", df_flow, family='poisson')

In [4]:
# IBM for glms
def IBM_1step_glm(zt, age, models, v_index, env, trace_fec, trace_flow, trace_sur, trace_grw_f, trace_grw_nf, 
                  tmax=np.nan, tmin=np.nan, precip=np.nan):
    # initialize 
    current_n = np.shape(zt)[0]
    rep_stalks = np.repeat(np.nan, current_n)[:, None]
    surv = np.repeat(np.nan, current_n)[:, None]
    zprime = np.repeat(np.nan, current_n)[:, None]
    ageprime = np.repeat(np.nan, current_n)[:, None]
    num_recruits = 0 
    zt = np.array(zt)[:, None]
    age = np.array(age)[:, None]

    if env==True:
        x_tmax = np.repeat(tmax, age.shape[0])
        x_tmin = np.repeat(tmin, age.shape[0])
        x_precip = np.repeat(precip, age.shape[0])
        X_t = np.concatenate((zt, age, x_tmax[:, None], x_tmin[:, None], x_precip[:, None]), axis=1)
    else: 
        X_t = np.concatenate((zt, age), axis=1)

    # we simulate those breeders first
    p_breeding = models['m_fec'].predict(trace_fec, data=pd.DataFrame(X_t, columns=['size', 'age', 'tmax', 'tmin', 'precip']), inplace=False)
    rep_breeding = np.random.binomial(n = 1, p = p_breeding.posterior['size_Next_mean'].values[0][v_index[0], : ])
    whether_bre = (rep_breeding == 1).reshape(1, current_n)[0]        
    num_bre = np.sum(whether_bre)
    num_nonbre = current_n - num_bre
        
    lambda_t = models["m_flow_poi"].predict(trace_flow, data=pd.DataFrame(X_t[whether_bre, :], columns=['size', 'age', 'tmax', 'tmin', 'precip']), inplace=False)
    lambda_t = lambda_t.posterior['size_Next_mean'].values[0][v_index[1], : ] 
    if np.any(np.isnan(lambda_t)):
        warnings.warn('nan lambda_t produced!!')
        print('nan lambda_t produced!!', flush=True)
        data = pd.DataFrame(0, index=[0], columns=['size', 'sizeNext', 'fec', 'flow', 'surv', 'age', 'ageNext']) 

        return(data)

    if np.any(np.array(lambda_t) >= 10000):
        warnings.warn('lambda_t is too large !!')
        print('lambda_t is too large !!', flush=True) 
        data = pd.DataFrame(0, index=[0], columns=['size', 'sizeNext', 'fec', 'flow', 'surv', 'age', 'ageNext']) 

        return(data)
    else:
        rep_stalks[whether_bre] = np.random.poisson(lambda_t)[:, None] + 1
           
    # simulate recruits based on the number of flowering stalks
    if (num_bre != 0):
        num_recruits = np.random.binomial(n = np.nansum(rep_stalks), p = models["recruit_p"])
        # assign size for those recruits
        rep_size = stats.gamma(a = models["alpha"], scale = 1/models["beta"]).rvs(size=num_recruits)[:, None]
            
        
    # now, simulating surviving.
    p_surv = models['m_sur'].predict(trace_sur, data=pd.DataFrame(X_t, columns=['size', 'age', 'tmax', 'tmin', 'precip']), inplace=False)
    surv = np.random.binomial(n = 1, p = p_surv.posterior['size_Next_mean'].values[0][v_index[2], : ])
    whether_surv = (surv == 1).reshape(1, current_n)[0]
    num_surv = np.sum(whether_surv)

    # let these survivors grow up
    # for breeders
    X_t_f = X_t[whether_surv & whether_bre]
    mean_zprime_f = models["m_grw_f"].predict(trace_grw_f, data=pd.DataFrame(X_t_f, columns=['size', 'age', 'tmax', 'tmin', 'precip']), inplace=False)
    zprime[whether_surv & whether_bre] = np.random.normal(mean_zprime_f.posterior['size_Next_mean'].values[0][v_index[3], : ], 
                                                          mean_zprime_f.posterior['size_Next_sigma'].values[0][v_index[3]])[:, None]
    
    # for non-breeders
    not_bre = whether_bre == False
    X_t_nf = X_t[whether_surv & not_bre]
    mean_zprime_nf = models["m_grw_nf"].predict(trace_grw_nf, data=pd.DataFrame(X_t_nf, columns=['size', 'age', 'tmax', 'tmin', 'precip']), inplace=False)
    zprime[whether_surv & not_bre] = np.random.normal(mean_zprime_nf.posterior['size_Next_mean'].values[0][v_index[4], : ], 
                                                      mean_zprime_nf.posterior['size_Next_sigma'].values[0][v_index[4]])[:, None]
    
    # in our case, age 8 is an absorbing state.
    ageprime[whether_surv] = np.minimum(age[whether_surv]+1, 8)
    
    
    rep_breeding = rep_breeding[:, None]
    surv = surv[:, None]
    # store the simulation data 
    # Here: age is for the gae at current not ageNext
    if (num_bre != 0):
        zprime = np.concatenate((zprime, rep_size))
        zt = np.concatenate((zt, np.repeat(np.nan, num_recruits)[:, None]))
        rep_breeding = np.concatenate((rep_breeding, np.repeat(np.nan, num_recruits)[:, None]))
        rep_stalks = np.concatenate((rep_stalks, np.repeat(np.nan, num_recruits)[:, None]))
        surv = np.concatenate((surv, np.repeat(np.nan, num_recruits)[:, None]))
        age = np.concatenate((age, np.repeat(np.nan, num_recruits)[:, None]))
        ageprime = np.concatenate((ageprime, np.repeat(1, num_recruits)[:, None]))
    data = pd.DataFrame(np.concatenate((zt, zprime, rep_breeding, rep_stalks, surv, age, ageprime), axis=1),
                        columns=['size', 'sizeNext', 'fec', 'flow', 'surv', 'age', 'ageNext']) 

    return(data)

In [5]:
models = {
    "m_sur": model_sur,
    "m_grw_f": model_grw_f,
    "m_grw_nf": model_grw_nf,
    "m_fec": model_fec,
    "m_flow_poi": model_flow
}

models['alpha'] = theta_mle_full['alpha'][0]
models['beta'] = theta_mle_full['beta'][0]
models['recruit_p'] = theta_mle_full['recruit_p'][0]

para_names_whole_glm = ['surv_int', 'surv_z', 'surv_age', 'surv_tmax', 'surv_tmin', 'surv_precip', 

                        'grow_int_f', 'grow_z_f', 'grow_age_f', 'grow_sd_f', 'grow_tmax_f', 'grow_tmin_f', 'grow_precip_f',
                
                        'grow_int_nf', 'grow_z_nf', 'grow_age_nf', 'grow_sd_nf', 'grow_tmax_nf',  'grow_tmin_nf', 'grow_precip_nf',
                
                        'fec_int', 'fec_z', 'fec_age', 'fec_tmax', 'fec_tmin', 'fec_precip',
                
                        'flow_int', 'flow_z', 'flow_age', 'flow_tmax', 'flow_tmin', 'flow_precip']

In [ ]:
# genetrate simulations for glms, long-term predictions, 5000 simulations
s = []; n_all = []; n1_all = []; n2_all = []
for _ in range(5000):
    c_s = []; n_size = []; n1_size = []; n2_size = []

    models['alpha'] = theta_mle_full['alpha'][0]
    models['beta'] = theta_mle_full['beta'][0]
    models['recruit_p'] = theta_mle_full['recruit_p'][0]

    r_index = np.random.randint(low=0, high=5000, size=5)
    data_simu = IBM_1step_glm(zt=popu_structure_noid_current(popu_0toT[0])[0], trace_fec=trace_fec, trace_flow=trace_flow, 
                              trace_sur=trace_sur, trace_grw_f=trace_grw_f, trace_grw_nf=trace_grw_nf, 
                              age = popu_structure_noid_current(popu_0toT[0])[1], models=models, env=True, 
                              v_index = r_index, 
                              tmax=w.iloc[0, 0], tmin=w.iloc[1, 0], precip=w.iloc[2, 0])


    current_s = list_comparisons_interested(exp_1step_data=popu_all[0], simu_1step_data=data_simu)

    c_s.append(current_s)
    n_new1 = np.sum(data_simu["surv"] == 1)
    n_new2 = np.sum(np.isnan(data_simu["size"])) 
    n_size.append(n_new1+n_new2); n1_size.append(n_new1); n2_size.append(n_new2) 

    for t in range(1, len(popu_all)):
        z = popu_structure_noid(data_simu) 

        models['alpha'] = theta_mle_full['alpha'][t]
        models['beta'] = theta_mle_full['beta'][t]
        models['recruit_p'] = theta_mle_full['recruit_p'][t]

        data_simu = IBM_1step_glm(zt=z[0], age=z[1], trace_fec=trace_fec, trace_flow=trace_flow, 
                                  trace_sur=trace_sur, trace_grw_f=trace_grw_f, trace_grw_nf=trace_grw_nf, 
                                  models=models, env=True, v_index = r_index, 
                                  tmax=w.iloc[0, t],  tmin=w.iloc[1, t], precip=w.iloc[2, t])

        
        # try:
        #     current_s = list_comparisons_interested(exp_1step_data=popu_all[t], simu_1step_data=data_simu)
        # except Exception as e:
        #     now = datetime.now().strftime("%H_%M_%S")
        #     pickle.dump(z , open(file = os.getcwd() + f"/error/z{t}" + now + ".pkl", mode="wb"))
        #     pickle.dump(data_simu , open(file = os.getcwd() + "/error/data_simu" + now + ".pkl", mode="wb"))
        #     pickle.dump(r_index, open(file = os.getcwd() + "/error/index" + now + ".pkl", mode="wb"))
        #     print(f'Error found at t={t}', flush=True)
        #     sys.exit(e)                     
        # c_s.append(current_s)
        n_new1 = np.sum(data_simu["surv"] == 1)
        n_new2 = np.sum(np.isnan(data_simu["size"])) 
        n_size.append(n_new1+n_new2); n1_size.append(n_new1); n2_size.append(n_new2) 

    # s.append(c_s)
    n_all.append(n_size)
    n1_all.append(n1_size)
    n2_all.append(n2_size)

# np.save(open(file = os.getcwd()+"/true_popu/simulations/glm_simulations", mode="wb"), n_all)

In [ ]:
# genetrate simulations for glms, 1-step predictions, 5000 simulations
s = []; n_all = []; n1_all = []; n2_all = []
for _ in range(5000):
    c_s = []; n_size = []; n1_size = []; n2_size = []

    models['alpha'] = theta_mle_full['alpha'][0]
    models['beta'] = theta_mle_full['beta'][0]
    models['recruit_p'] = theta_mle_full['recruit_p'][0]

    r_index = np.random.randint(low=0, high=5000, size=5)
    data_simu = IBM_1step_glm(zt=popu_structure_noid_current(popu_0toT[0])[0], trace_fec=trace_fec, trace_flow=trace_flow, 
                              trace_sur=trace_sur, trace_grw_f=trace_grw_f, trace_grw_nf=trace_grw_nf, 
                              age = popu_structure_noid_current(popu_0toT[0])[1], models=models, env=True, 
                              v_index = r_index, 
                              tmax=w.iloc[0, 0], tmin=w.iloc[1, 0], precip=w.iloc[2, 0])


    current_s = list_comparisons_interested(exp_1step_data=popu_all[0], simu_1step_data=data_simu)

    c_s.append(current_s)
    n_new1 = np.sum(data_simu["surv"] == 1)
    n_new2 = np.sum(np.isnan(data_simu["size"])) 
    n_size.append(n_new1+n_new2); n1_size.append(n_new1); n2_size.append(n_new2) 

    for t in range(1, len(popu_all)):
        z =  popu_structure_noid_current(popu_all[t])

        models['alpha'] = theta_mle_full['alpha'][t]
        models['beta'] = theta_mle_full['beta'][t]
        models['recruit_p'] = theta_mle_full['recruit_p'][t]

        data_simu = IBM_1step_glm(zt=z[0], age=z[1], trace_fec=trace_fec, trace_flow=trace_flow, 
                                  trace_sur=trace_sur, trace_grw_f=trace_grw_f, trace_grw_nf=trace_grw_nf, 
                                  models=models, env=True, v_index = r_index, 
                                  tmax=w.iloc[0, t],  tmin=w.iloc[1, t], precip=w.iloc[2, t])

        
        # try:
        #     current_s = list_comparisons_interested(exp_1step_data=popu_all[t], simu_1step_data=data_simu)
        # except Exception as e:
        #     now = datetime.now().strftime("%H_%M_%S")
        #     pickle.dump(z , open(file = os.getcwd() + f"/error/z{t}" + now + ".pkl", mode="wb"))
        #     pickle.dump(data_simu , open(file = os.getcwd() + "/error/data_simu" + now + ".pkl", mode="wb"))
        #     pickle.dump(r_index, open(file = os.getcwd() + "/error/index" + now + ".pkl", mode="wb"))
        #     print(f'Error found at t={t}', flush=True)
        #     sys.exit(e)                     
        # c_s.append(current_s)
        n_new1 = np.sum(data_simu["surv"] == 1)
        n_new2 = np.sum(np.isnan(data_simu["size"])) 
        n_size.append(n_new1+n_new2); n1_size.append(n_new1); n2_size.append(n_new2) 

    # s.append(c_s)
    n_all.append(n_size)
    n1_all.append(n1_size)
    n2_all.append(n2_size)

# np.save(open(file = os.getcwd()+"/true_popu/simulations/glm_simulations_z", mode="wb"), n_all)

In [ ]:
# generate Long-term forecasting only for testing years (2009-2012), 
# given true population structure at 2008.
s = []; n_all = []; n1_all = []; n2_all = []
for _ in range(5000):
    c_s = []; n_size = []; n1_size = []; n2_size = []

    models['alpha'] = theta_mle_full['alpha'][5]
    models['beta'] = theta_mle_full['beta'][5]
    models['recruit_p'] = theta_mle_full['recruit_p'][5]

    r_index = np.random.randint(low=0, high=5000, size=5)
    data_simu = IBM_1step_glm(zt=popu_structure_noid_current(popu_all[5])[0], trace_fec=trace_fec, trace_flow=trace_flow, 
                              trace_sur=trace_sur, trace_grw_f=trace_grw_f, trace_grw_nf=trace_grw_nf, 
                              age = popu_structure_noid_current(popu_all[5])[1], models=models, env=True, 
                              v_index = r_index, 
                              tmax=w.iloc[0, 5], tmin=w.iloc[1, 5], precip=w.iloc[2, 5])


    current_s = list_comparisons_interested(exp_1step_data=popu_all[0], simu_1step_data=data_simu)

    c_s.append(current_s)
    n_new1 = np.sum(data_simu["surv"] == 1)
    n_new2 = np.sum(np.isnan(data_simu["size"])) 
    n_size.append(n_new1+n_new2); n1_size.append(n_new1); n2_size.append(n_new2) 

    for t in range(1, 4):
        z = popu_structure_noid(data_simu) 
        models['alpha'] = theta_mle_full['alpha'][5+t]
        models['beta'] = theta_mle_full['beta'][5+t]
        models['recruit_p'] = theta_mle_full['recruit_p'][5+t]

        data_simu = IBM_1step_glm(zt=z[0], age=z[1], trace_fec=trace_fec, trace_flow=trace_flow, 
                                  trace_sur=trace_sur, trace_grw_f=trace_grw_f, trace_grw_nf=trace_grw_nf, 
                                  models=models, env=True, v_index = r_index, 
                                  tmax=w.iloc[0, 5+t],  tmin=w.iloc[1, 5+t], precip=w.iloc[2, 5+t])

        
        n_new1 = np.sum(data_simu["surv"] == 1)
        n_new2 = np.sum(np.isnan(data_simu["size"])) 
        n_size.append(n_new1+n_new2); n1_size.append(n_new1); n2_size.append(n_new2) 


    n_all.append(n_size)
    n1_all.append(n1_size)
    n2_all.append(n2_size)

np.save(open(file = os.getcwd()+"/true_popu/simulations/glm_simulations_testingyears.pkl", mode="wb"), n_all)